In [1]:
# In[1]:


# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print((os.listdir("../input")))

# Any results you write to the current directory are saved as output.

In [2]:
# In[2]:


train_df=pd.read_csv("../input/train.csv")
test_df=pd.read_csv("../input/test.csv")

In [3]:
# In[3]:


print((train_df.info()))
print(("--"*20))
print((test_df.info()))

In [4]:
# In[4]:


X=train_df.drop("Survived",axis=1)
Y=train_df["Survived"]

In [ ]:
# In[5]:


# Leakage fix: train and test are NOT concatenated before preprocessing anymore.
# Previously imputation medians/modes, Fare quartile edges, Age bin edges and label
# encodings were all computed on train+test combined, leaking test-set information
# into the training features. All of these are now learned from training rows only
# (see TitanicPreprocessor below) and then applied unchanged to the test set.

In [ ]:
# In[6]:


print((X.info()))


# # Data Cleaning
# * Correct
# * Complete
# * Create
# * Convert

In [ ]:
# In[7]:


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder


class TitanicPreprocessor(BaseEstimator, TransformerMixin):
    """Imputation, feature creation, binning and encoding.

    Every data-dependent statistic is learned in fit() (on training data only)
    and reused as-is in transform(), so test/validation rows never influence it.
    """
    features=["Pclass","Family_size","Is_alone","Sex_encode","Age_encode","Fare_encode","Embarked_encode"]

    def _impute(self, df):
        df=df.copy()
        df["Age"]=self.age_imputer_.transform(df[["Age"]]).ravel()
        df["Fare"]=self.fare_imputer_.transform(df[["Fare"]]).ravel()
        df["Embarked"]=self.embarked_imputer_.transform(df[["Embarked"]]).ravel()
        df.drop(["Cabin"],axis=1,inplace=True)
        # Row-wise features (no fitted statistics involved)
        df["Family_size"]=df["SibSp"]+df["Parch"]+1
        df["Is_alone"]=(df["Family_size"]==1).astype(int)
        return df

    def fit(self, X, y=None):
        self.age_imputer_=SimpleImputer(strategy="median").fit(X[["Age"]])
        self.fare_imputer_=SimpleImputer(strategy="median").fit(X[["Fare"]])
        self.embarked_imputer_=SimpleImputer(strategy="most_frequent").fit(X[["Embarked"]])
        df=self._impute(X)
        # Bin edges learned from training data only; outer edges opened up so
        # unseen values outside the training range still fall into a bin.
        _,fare_edges=pd.qcut(df["Fare"],4,retbins=True)
        _,age_edges=pd.cut(df["Age"].astype(int),4,retbins=True)
        fare_edges[0],fare_edges[-1]=-np.inf,np.inf
        age_edges[0],age_edges[-1]=-np.inf,np.inf
        self.fare_edges_=fare_edges
        self.age_edges_=age_edges
        self.sex_encoder_=LabelEncoder().fit(df["Sex"])
        self.embarked_encoder_=LabelEncoder().fit(df["Embarked"])
        return self

    def transform(self, X):
        df=self._impute(X)
        df["Fare_bin"]=pd.cut(df["Fare"],self.fare_edges_)
        df["Age_bin"]=pd.cut(df["Age"].astype(int),self.age_edges_)
        df["Sex_encode"]=self.sex_encoder_.transform(df["Sex"])
        df["Age_encode"]=pd.cut(df["Age"].astype(int),self.age_edges_,labels=False)
        df["Fare_encode"]=pd.cut(df["Fare"],self.fare_edges_,labels=False)
        df["Embarked_encode"]=self.embarked_encoder_.transform(df["Embarked"])
        return df[self.features]

In [ ]:
# In[14]:


# Fit preprocessing on the training set only, then apply it to the test set
preprocessor=TitanicPreprocessor()
X_train=preprocessor.fit_transform(X)
X_test=preprocessor.transform(test_df)

In [15]:
# In[15]:


X_train.head()


# # Model Creation

In [16]:
# In[16]:


from sklearn import ensemble, neural_network , neighbors , svm , linear_model ,naive_bayes ,tree , gaussian_process,discriminant_analysis
from xgboost import XGBClassifier, XGBRFClassifier

In [17]:
# In[17]:


#Machine Learning Algorithms
MLA=[
    #Ensemble Methods
    ensemble.AdaBoostClassifier(),
    ensemble.BaggingClassifier(),
    ensemble.ExtraTreesClassifier(),
    ensemble.GradientBoostingClassifier(),
    ensemble.RandomForestClassifier(),

    #Gaussian Processes
    gaussian_process.GaussianProcessClassifier(),
    
    #GLM
    linear_model.LogisticRegressionCV(),
    linear_model.PassiveAggressiveClassifier(),
    linear_model.RidgeClassifierCV(),
    linear_model.SGDClassifier(),
    linear_model.Perceptron(),
    
    #Navies Bayes
    naive_bayes.BernoulliNB(),
    naive_bayes.GaussianNB(),
    
    #Nearest Neighbor
    neighbors.KNeighborsClassifier(),
    
    #SVM
    svm.SVC(probability=True),
    svm.NuSVC(probability=True),
    svm.LinearSVC(),
    
    #Trees    
    tree.DecisionTreeClassifier(),
    tree.ExtraTreeClassifier(),
    
    #Discriminant Analysis
    discriminant_analysis.LinearDiscriminantAnalysis(),
    discriminant_analysis.QuadraticDiscriminantAnalysis(),

    #Neural Network
    neural_network.MLPClassifier(),
    
    #xgboost
    XGBClassifier(),
    XGBRFClassifier()
]

In [18]:
# In[18]:


MLA_compare=["Algorithm_Name","Parameters", "Mean_Accuracy","Standard_Deviation"]
MLA_compare=pd.DataFrame(columns=MLA_compare)

In [19]:
# In[19]:


from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

index=0
for algo in MLA:
    clf=algo
    clf.fit(X_train,Y)
    # Preprocessing is refit inside each CV fold so validation folds do not leak into it
    cv=cross_val_score(estimator=make_pipeline(TitanicPreprocessor(),clone(algo)),X=X,y=Y,cv=10,n_jobs=-1)
    cv_mean=cv.mean()
    cv_std=cv.std()
    MLA_compare.loc[index,"Algorithm_Name"]=algo.__class__.__name__
    MLA_compare.loc[index,"Parameters"]=str(algo.get_params())
    MLA_compare.loc[index,"Mean_Accuracy"]=cv_mean
    MLA_compare.loc[index,"Standard_Deviation"]=cv_std
    index+=1

In [20]:
# In[20]:


MLA_compare


# ## Fine Tuning Models by tuning Hyper parameters

In [21]:
# In[21]:


selected_model=svm.NuSVC(probability=True)
selected_model.fit(X_train,Y)

In [22]:
# In[22]:


# Parameters=[{"base_score":[0.1,0.2,0.3,0.5,0.6,0.9,1,2],
#              "gamma":[0,0.1,0.2,0.3,0.4,0.5,1],
#              "learning_rate":[0,0.1,0.2,0.3,0.4,1],
#             }]

In [23]:
# In[23]:


# from sklearn.model_selection import GridSearchCV
# grid=GridSearchCV(estimator=selected_model,param_grid=Parameters,n_jobs=-1,cv=10,scoring="accuracy")
# result=grid.fit(X=X_train,y=Y)

In [24]:
# In[24]:


# best_accuracy=result.best_score_
# best_parameters=result.best_params_

In [25]:
# In[25]:


y_pred=selected_model.predict(X_test)
y_pred=pd.DataFrame({"PassengerId":test_df["PassengerId"],"Survived":y_pred})
y_pred.to_csv("Submit.csv",index=False)

In [26]:
# In[26]:


test_df

In [ ]:
# In[ ]:

In [ ]:
# In[ ]: